In [1]:
import warnings
warnings.filterwarnings("ignore")

from __future__ import annotations
import os
from io import StringIO
from pathlib import Path
from typing import Dict, Any, List, Optional
from datetime import datetime, timezone
from src.config import get_secret
import numpy as np
import pandas as pd
import requests
import pytz

# ── Constants ─────────────────────────────────────────────────────────────────
YEAR: int = 2026
TOUR_KEYS: List[str] = ["PGA", "EURO", "LIV"]
EXCLUDED_EVENT_IDS: set = {18}

DG_ROUNDS_URL = "https://feeds.datagolf.com/historical-raw-data/rounds"
DG_FIELD_URL  = "https://feeds.datagolf.com/field-updates"
DG_ODDS_URL   = "https://feeds.datagolf.com/betting-tools/outrights"

DG_API_KEY = get_secret("DATAGOLF_API_KEY")
if not DG_API_KEY:
    raise RuntimeError("Missing DATAGOLF_API_KEY env var.")

PROJECT_ROOT = Path("/Users/joshmacbook/python_projects/GolfStats")
DATA_ROOT    = PROJECT_ROOT / "Data"
RAW_DIR      = DATA_ROOT / "Raw"
CLEAN_DIR    = DATA_ROOT / "Clean" / "Combined"
INUSE_DIR    = DATA_ROOT / "in Use"

for _p in [RAW_DIR, CLEAN_DIR, INUSE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

CLEAN_YEAR_PATH     = CLEAN_DIR / f"combined_rounds_{YEAR}.csv"
INUSE_ALL_PATH      = INUSE_DIR / "combined_rounds_all_2017_2026.csv"
FINISHES_PATH       = INUSE_DIR / "Finishes.csv"
THIS_WEEK_FIELD_CSV = INUSE_DIR / "this_week_field.csv"
THIS_WEEK_ODDS_CSV  = INUSE_DIR / "this_week_odds.csv"
FIELDS_XLSX         = INUSE_DIR / "Fields.xlsx"
ALL_PLAYERS_XLSX    = INUSE_DIR / "All_players.xlsx"
SCHED_XLSX          = INUSE_DIR / "OAD_2026_Schedule.xlsx"

TOURS: Dict[str, Dict[str, str]] = {
    "PGA":  {"folder": "PGA",  "prefix": "PGA",  "api_tour": "PGA"},
    "EURO": {"folder": "EURO", "prefix": "EURO", "api_tour": "euro"},
    "LIV":  {"folder": "LIV",  "prefix": "liv",  "api_tour": "liv"},
}

TIMEOUT = 60
SESSION = requests.Session()
ET      = pytz.timezone("America/New_York")

# ── Utilities ─────────────────────────────────────────────────────────────────
def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def coerce_int(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df

def safe_to_datetime(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce")

def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path, **kwargs)

def write_csv(path: Path, df: pd.DataFrame) -> None:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)

def read_excel(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_excel(path)

def write_excel(path: Path, df: pd.DataFrame, *, sheet_name: str = "Sheet1") -> None:
    ensure_dir(path.parent)
    with pd.ExcelWriter(path, engine="openpyxl", mode="w") as w:
        df.to_excel(w, index=False, sheet_name=sheet_name)

def pull_csv_to_df(url: str, params: Dict[str, Any], *, save_path: Optional[Path] = None) -> pd.DataFrame:
    resp = SESSION.get(url, params=params, timeout=TIMEOUT)
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    if save_path is not None:
        ensure_dir(save_path.parent)
        df.to_csv(save_path, index=False)
        print(f"[PULL] Saved -> {save_path} | rows: {len(df):,}")
    return df

# ── Cleaning helpers ──────────────────────────────────────────────────────────
NON_NUMERIC_FINISH = {"CUT", "WD", "DQ", "MDF", "NAN"}

def clean_finish(val):
    v = str(val).upper().strip()
    if v in NON_NUMERIC_FINISH:
        return v
    if v.startswith("T") and v[1:].isdigit():
        return int(v[1:])
    if v.isdigit():
        return int(v)
    return v

# ── LIV event_id patch map ────────────────────────────────────────────────────
LIV_EVENT_ID_MAP = {k.strip().lower(): v for k, v in {
    "adelaide": 1012, "andalucia": 1024, "bangkok": 1006, "bedminster": 1003,
    "boston": 1004, "chicago": 1005, "dallas": 1031,
    "dallas (team finalstroke play)": 1026, "dc": 1015, "greenbrier": 1017,
    "hong kong": 1020, "houston": 1022, "indianapolis": 1032, "jeddah": 1007,
    "korea": 1029, "las vegas": 1019, "london": 1001, "mayakoba": 1009,
    "mexico city": 1028, "miami": 1021, "miami (team finalstroke play)": 1008,
    "michigan (team finalstroke play)": 1033, "nashville": 1023, "orlando": 1011,
    "portland": 1002, "promotions event": 1018, "riyadh": 1027, "singapore": 1013,
    "tucson": 1010, "tulsa": 1014, "united kingdom": 1025, "valderrama": 1016,
    "virginia": 1030,
}.items()}

def patch_liv_ids(df: pd.DataFrame) -> pd.DataFrame:
    if not {"tour", "event_name"}.issubset(df.columns):
        return df
    out = df.copy()
    out["_tour"]  = out["tour"].astype(str).str.strip().str.upper()
    out["_ename"] = out["event_name"].astype(str).str.strip().str.lower()
    target = out["_tour"].eq("LIV") & out["_ename"].isin(LIV_EVENT_ID_MAP)
    if target.any():
        new_ids = out.loc[target, "_ename"].map(LIV_EVENT_ID_MAP).astype("Int64")
        for col in ["event_id", "course_num"]:
            if col in out.columns:
                out.loc[target, col] = new_ids
        print(f"[PATCH] LIV id/course_num updates: {int(target.sum()):,}")
    return out.drop(columns=["_tour", "_ename"], errors="ignore")

# ── round_date derivation ─────────────────────────────────────────────────────
def add_round_date(df: pd.DataFrame) -> pd.DataFrame:
    needed = {"event_completed", "round_num", "tour", "event_name"}
    if not needed.issubset(df.columns):
        return df
    out = df.copy()
    out["_ec_dt"] = safe_to_datetime(out["event_completed"])
    if out["_ec_dt"].isna().all():
        return df
    out["round_num"] = pd.to_numeric(out["round_num"], errors="coerce")
    max_round = out.groupby(["tour", "event_name", "_ec_dt"])["round_num"].transform("max")
    out["round_date"] = (
        out["_ec_dt"] - pd.to_timedelta(max_round - out["round_num"], unit="D")
    ).dt.date.astype(str)
    return out.drop(columns=["_ec_dt"], errors="ignore")

# ── replace year in csv ───────────────────────────────────────────────────────
def replace_year_in_csv(path: Path, new_df: pd.DataFrame, *, year_col: str = "year", year_val: int = YEAR) -> None:
    if path.exists():
        old = pd.read_csv(path, low_memory=False)
        old[year_col] = pd.to_numeric(old.get(year_col), errors="coerce")
        out = pd.concat([old[old[year_col] != year_val], new_df], ignore_index=True)
    else:
        out = new_df
    out.to_csv(path, index=False)
    print(f"[REPLACE] {path.name} | replaced {year_val} | rows: {len(out):,}")

# ── round positions ───────────────────────────────────────────────────────────
def add_round_positions(df: pd.DataFrame) -> pd.DataFrame:
    for col in ["to_par", "cum_to_par", "round_position", "round_position_text"]:
        if col in df.columns:
            df = df.drop(columns=[col])
    df = df.copy()
    df["to_par"] = df["round_score"] - df["course_par"]
    df = df.sort_values(["event_id", "season", "dg_id", "round_num"]).reset_index(drop=True)
    df["cum_to_par"] = df.groupby(["event_id", "season", "dg_id"])["to_par"].cumsum()

    missing_par  = df.groupby(["event_id", "season"])["course_par"].apply(lambda x: x.isna().all())
    missing_keys = missing_par[missing_par].index
    if len(missing_keys) > 0:
        mask = df.set_index(["event_id", "season"]).index.isin(missing_keys)
        df.loc[mask, "cum_to_par"] = (
            df[mask].groupby(["event_id", "season", "dg_id"])["round_score"].cumsum().values
        )

    def rank_group(g):
        valid = g["cum_to_par"].notna()
        pos   = pd.Series(np.nan, index=g.index)
        if valid.any():
            pos[valid] = g.loc[valid, "cum_to_par"].rank(method="min", ascending=True).astype(int)
        return pos

    df["round_position"] = (
        df.groupby(["event_id", "season", "round_num"], group_keys=False)
        .apply(lambda g: rank_group(g), include_groups=False)
    )
    df["round_position"] = pd.array(
        df["round_position"].where(df["round_position"].isna(), df["round_position"].astype("Int64")),
        dtype="Int64",
    )
    tie_counts = df.groupby(["event_id", "season", "round_num", "round_position"])["dg_id"].transform("count")
    df["round_position_text"] = df["round_position"].astype(str)
    df.loc[df["round_position"].isna(), "round_position_text"] = np.nan
    df.loc[tie_counts > 1, "round_position_text"] = "T" + df.loc[tie_counts > 1, "round_position"].astype(str)
    return df

# ── Finishes builder ──────────────────────────────────────────────────────────
def build_finishes(full_path: Path, out_path: Path, year: int = YEAR) -> None:
    df = pd.read_csv(full_path, low_memory=False)
    df["year"] = pd.to_numeric(df.get("season", df.get("year")), errors="coerce").astype("Int64")
    df = df[df["year"] == year].copy()
    df = coerce_int(df, ["event_id", "dg_id"])
    df["finish_num"] = pd.to_numeric(df.get("finish_num"), errors="coerce")
    sort_cols = [c for c in ["year", "event_id", "dg_id", "round_num", "round_date"] if c in df.columns]
    final = df.sort_values(sort_cols).groupby(["year", "event_id", "dg_id"], as_index=False).tail(1).copy()
    fn = final["finish_num"]
    final["made_cut"] = fn.notna().astype(int)
    final["CUT"]      = (1 - final["made_cut"]).astype(int)
    final["win"]      = ((fn == 1)  & fn.notna()).astype(int)
    final["top_5"]    = ((fn <= 5)  & fn.notna()).astype(int)
    final["top_10"]   = ((fn <= 10) & fn.notna()).astype(int)
    final["top_25"]   = ((fn <= 25) & fn.notna()).astype(int)
    if "finish_text" not in final.columns and "fin_text" in final.columns:
        final["finish_text"] = final["fin_text"]
    cols = ["year", "event_name", "event_id", "event_completed",
            "player_name", "dg_id", "finish_text", "finish_num",
            "win", "top_5", "top_10", "top_25", "made_cut", "CUT"]
    for c in cols:
        if c not in final.columns:
            final[c] = pd.NA
    out = final[cols].sort_values(["event_id", "finish_num", "player_name"], na_position="last")
    out.to_csv(out_path, index=False)
    print(f"[FINISHES] {len(out):,} rows for {year} -> {out_path.name}")

print("Setup complete.")


Setup complete.


In [2]:
# ============================================================
# BLOCK 1: Weekly PGA Field + Odds -> Fields.xlsx + All_players.xlsx
# ============================================================

NOW_ET = pd.Timestamp.now(tz=ET)

def _schedule_start_et(sched_df, event_id):
    row = sched_df.loc[sched_df["event_id"] == event_id]
    if row.empty:
        raise RuntimeError(f"Schedule has no row for event_id={event_id}")
    start = pd.to_datetime(row["start_date"].iloc[0], errors="coerce")
    if pd.isna(start):
        raise RuntimeError(f"Schedule start_date is NaT for event_id={event_id}")
    return pd.Timestamp(start.date()).tz_localize(ET)

def _ensure_fields_cols(df):
    out = df.copy()
    for c in ["year", "event_name", "event_id", "event_completed",
              "player_name", "dg_id", "close_odds", "field_pulled_at", "odds_pulled_at"]:
        if c not in out.columns:
            out[c] = pd.NA
    return coerce_int(out, ["year", "event_id", "dg_id"])

# 1) Pull field
field_raw = pull_csv_to_df(DG_FIELD_URL, {"tour": "PGA", "file_format": "csv", "key": DG_API_KEY},
                           save_path=THIS_WEEK_FIELD_CSV)
field_raw = coerce_int(field_raw, ["event_id", "dg_id"])
field = field_raw[["event_name", "event_id", "dg_id", "player_name", "owgr_rank"]].dropna(
    subset=["event_id", "dg_id", "player_name"]).copy()

ids = sorted(field["event_id"].dropna().unique().tolist())
if len(ids) != 1:
    raise RuntimeError(f"Expected 1 event_id in field feed; got {ids}")
FIELD_EVENT_ID   = int(ids[0])
FIELD_EVENT_NAME = field["event_name"].iloc[0]
print(f"[FIELD] event_id={FIELD_EVENT_ID} | {FIELD_EVENT_NAME} | players={len(field):,}")

# 2) Pull odds
try:
    odds_raw = pull_csv_to_df(DG_ODDS_URL,
                              {"tour": "pga", "market": "win", "odds_format": "decimal",
                               "file_format": "csv", "key": DG_API_KEY},
                              save_path=THIS_WEEK_ODDS_CSV)
except Exception as e:
    if THIS_WEEK_ODDS_CSV.exists():
        print(f"[ODDS] pull failed; falling back to cached file. Error: {e}")
        odds_raw = pd.read_csv(THIS_WEEK_ODDS_CSV)
    else:
        raise

odds_raw      = coerce_int(odds_raw, ["event_id", "dg_id"])
ODDS_EVENT_ID = None
if "event_id" in odds_raw.columns:
    oids = sorted(pd.to_numeric(odds_raw["event_id"], errors="coerce").dropna().astype(int).unique().tolist())
    if len(oids) == 1:
        ODDS_EVENT_ID = int(oids[0])
    else:
        print(f"[ODDS] ambiguous event_ids {oids} -- skipping close_odds write")
else:
    print("[ODDS] no event_id column -- skipping close_odds write")

close_odds_by_id = dict(zip(
    odds_raw["dg_id"].dropna().astype(int),
    pd.to_numeric(odds_raw["datagolf_base_history_fit"], errors="coerce")
))
print(f"[ODDS] mapped_ids={len(close_odds_by_id):,} | ODDS_EVENT_ID={ODDS_EVENT_ID}")

# 3) Load schedule
sched = read_excel(SCHED_XLSX)
sched["event_id"]   = pd.to_numeric(sched["event_id"], errors="coerce").astype("Int64")
sched["start_date"] = pd.to_datetime(sched["start_date"], errors="coerce")
FIELD_START_ET = _schedule_start_et(sched, FIELD_EVENT_ID)
field_started  = NOW_ET >= FIELD_START_ET
print(f"[SCHED] event_id={FIELD_EVENT_ID} start={FIELD_START_ET} | started={field_started}")

# 4) Update Fields.xlsx
fields = _ensure_fields_cols(read_excel(FIELDS_XLSX))
mask   = (fields["year"] == YEAR) & (fields["event_id"] == FIELD_EVENT_ID)

existing_odds = dict(zip(
    fields.loc[mask, "dg_id"].dropna().astype(int),
    fields.loc[mask, "close_odds"]
))

wk = field[["event_name", "event_id", "dg_id", "player_name"]].copy()
wk["year"]            = YEAR
wk["event_completed"] = pd.Timestamp(FIELD_START_ET.date())
wk["field_pulled_at"] = NOW_ET.strftime("%Y-%m-%d %H:%M:%S %Z")

if field_started:
    print("[FIELDS] Event started -- freezing close_odds")
    wk["close_odds"]     = wk["dg_id"].astype(int).map(existing_odds)
    wk["odds_pulled_at"] = pd.NA
else:
    print("[FIELDS] Event not started -- updating close_odds")
    wk["close_odds"]     = wk["dg_id"].astype(int).map(close_odds_by_id)
    wk["odds_pulled_at"] = NOW_ET.strftime("%Y-%m-%d %H:%M:%S %Z")

wk = wk[["year", "event_name", "event_id", "event_completed",
          "player_name", "dg_id", "close_odds", "field_pulled_at", "odds_pulled_at"]]
fields_new = pd.concat([fields.loc[~mask], wk], ignore_index=True)
fields_new["player_name"] = fields_new["player_name"].astype(str)
fields_new["event_name"]  = fields_new["event_name"].astype(str)
write_excel(FIELDS_XLSX, fields_new)
print(f"[FIELDS] Written | dropped={int(mask.sum()):,} | added={len(wk):,} | total={len(fields_new):,}")

# Optional: update close_odds for next event if it hasn't started and already has a field block
if ODDS_EVENT_ID is not None and ODDS_EVENT_ID != FIELD_EVENT_ID:
    try:
        odds_start   = _schedule_start_et(sched, ODDS_EVENT_ID)
        odds_mask    = (fields_new["year"] == YEAR) & (fields_new["event_id"] == ODDS_EVENT_ID)
        has_block    = bool(odds_mask.any())
        odds_started = NOW_ET >= odds_start
        print(f"[ODDS->FIELDS] event_id={ODDS_EVENT_ID} started={odds_started} has_block={has_block}")
        if not odds_started and has_block:
            fields_new.loc[odds_mask, "close_odds"] = (
                fields_new.loc[odds_mask, "dg_id"].astype("Int64")
                .map(lambda x: close_odds_by_id.get(int(x), pd.NA) if pd.notna(x) else pd.NA)
            )
            fields_new.loc[odds_mask, "odds_pulled_at"] = NOW_ET.strftime("%Y-%m-%d %H:%M:%S %Z")
            write_excel(FIELDS_XLSX, fields_new)
            print("[ODDS->FIELDS] Updated close_odds for next event")
        elif odds_started:
            print("[ODDS->FIELDS] Already started -- not updating")
        else:
            print("[ODDS->FIELDS] No field block yet -- not updating")
    except Exception as e:
        print(f"[ODDS->FIELDS] Skipped: {e}")

# 5) Update owgr in All_players.xlsx
ap = read_excel(ALL_PLAYERS_XLSX)
if "owgr" not in ap.columns:
    ap["owgr"] = pd.NA
ap["dg_id"] = pd.to_numeric(ap["dg_id"], errors="coerce").astype("Int64")
owgr_map = dict(zip(field["dg_id"].dropna().astype(int), field["owgr_rank"]))
ap["owgr"] = ap.apply(
    lambda r: owgr_map.get(int(r["dg_id"]), r["owgr"]) if pd.notna(r["dg_id"]) else r["owgr"], axis=1
)
write_excel(ALL_PLAYERS_XLSX, ap)
print(f"[ALL_PLAYERS] owgr updated for {len(owgr_map):,} players")


[PULL] Saved -> /Users/joshmacbook/python_projects/GolfStats/Data/in Use/this_week_field.csv | rows: 123
[FIELD] event_id=11 | THE PLAYERS Championship | players=123
[PULL] Saved -> /Users/joshmacbook/python_projects/GolfStats/Data/in Use/this_week_odds.csv | rows: 123
[ODDS] no event_id column -- skipping close_odds write
[ODDS] mapped_ids=123 | ODDS_EVENT_ID=None
[SCHED] event_id=11 start=2026-03-12 00:00:00-04:00 | started=False
[FIELDS] Event not started -- updating close_odds
[FIELDS] Written | dropped=123 | added=123 | total=1,528
[ALL_PLAYERS] owgr updated for 123 players


In [3]:
# ============================================================
# BLOCK 2: Weekly rounds refresh -> combined_rounds_all_2017_2026.csv
# ============================================================

# 1) Pull fresh 2026 rounds for each tour
for tour_key in TOUR_KEYS:
    cfg      = TOURS[tour_key]
    out_path = ensure_dir(RAW_DIR / cfg["folder"]) / f"{cfg['prefix']}_rounds_{YEAR}.csv"
    params   = {"tour": cfg["api_tour"], "event_id": "all", "year": YEAR,
                "file_format": "csv", "key": DG_API_KEY}
    df = pull_csv_to_df(DG_ROUNDS_URL, params, save_path=out_path)
    coerce_int(df, ["event_id", "dg_id", "year", "round_num"]).to_csv(out_path, index=False)

# 2) Stack tours
dfs = []
for tour_key in TOUR_KEYS:
    cfg   = TOURS[tour_key]
    fpath = RAW_DIR / cfg["folder"] / f"{cfg['prefix']}_rounds_{YEAR}.csv"
    df    = read_csv(fpath)
    df["tour"] = tour_key
    df["year"] = YEAR
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
combined = coerce_int(combined, ["event_id", "dg_id", "year", "round_num"])

# 3) finish_num
if "fin_text" in combined.columns:
    combined["fin_text"]   = combined["fin_text"].astype(str).str.upper().str.strip()
    combined["finish_num"] = combined["fin_text"].apply(clean_finish)
    print("[CLEAN] finish_num derived from fin_text")

# 4) Patch LIV ids
combined = patch_liv_ids(combined)

# 5) Drop excluded events
before   = len(combined)
combined = combined[~combined["event_id"].isin(EXCLUDED_EVENT_IDS)].copy()
print(f"[FILTER] Excluded {EXCLUDED_EVENT_IDS} | dropped {before - len(combined):,} rows")

# 6) Derive round_date
combined = add_round_date(combined)

# 7) Write clean year file
write_csv(CLEAN_YEAR_PATH, combined)
print(f"[CLEAN] {CLEAN_YEAR_PATH.name} | rows={len(combined):,}")

# 8) Replace 2026 in full combined file
replace_year_in_csv(INUSE_ALL_PATH, combined)

# 9) Add round positions to full file
print("[POSITIONS] Computing on full file...")
full_df = read_csv(INUSE_ALL_PATH, low_memory=False)
full_df = add_round_positions(full_df)
nan_pos = full_df["round_position"].isna().sum()
if nan_pos > 0:
    top = full_df[full_df["round_position"].isna()]["event_name"].value_counts()
    print(f"  {nan_pos:,} NaN positions (expected for team-format rounds):")
    for evt, cnt in top.items():
        print(f"    {evt}: {cnt}")
write_csv(INUSE_ALL_PATH, full_df)
print(f"[POSITIONS] Done | {len(full_df):,} rows, {len(full_df.columns)} cols")

# 10) Build Finishes.csv
build_finishes(INUSE_ALL_PATH, FINISHES_PATH)

print("\n[DONE] Weekly refresh complete.")


[PULL] Saved -> /Users/joshmacbook/python_projects/GolfStats/Data/Raw/PGA/PGA_rounds_2026.csv | rows: 3,318
[PULL] Saved -> /Users/joshmacbook/python_projects/GolfStats/Data/Raw/EURO/EURO_rounds_2026.csv | rows: 2,804
[PULL] Saved -> /Users/joshmacbook/python_projects/GolfStats/Data/Raw/LIV/liv_rounds_2026.csv | rows: 835
[CLEAN] finish_num derived from fin_text
[PATCH] LIV id/course_num updates: 835
[FILTER] Excluded {18} | dropped 0 rows
[CLEAN] combined_rounds_2026.csv | rows=6,957
[REPLACE] combined_rounds_all_2017_2026.csv | replaced 2026 | rows: 317,073
[POSITIONS] Computing on full file...
[POSITIONS] Done | 317,073 rows, 41 cols
[FINISHES] 2,340 rows for 2026 -> Finishes.csv

[DONE] Weekly refresh complete.


In [4]:
# ============================================================
# BLOCK 3: Approach skill refresh (run as needed)
# ============================================================

PERIODS     = ["l12", "l24", "ytd"]
BASE_URL    = "https://feeds.datagolf.com/preds/approach-skill"
OUTPUT_PATH = INUSE_DIR / "approach_skill_all_periods.csv"

def fetch_approach_period(period: str) -> pd.DataFrame:
    resp    = requests.get(BASE_URL, params={"period": period, "file_format": "json", "key": DG_API_KEY}, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    if isinstance(payload, list):
        rows, last_updated = payload, None
    else:
        rows         = payload.get("players") or payload.get("data") or payload.get("results") or []
        last_updated = payload.get("last_updated") or payload.get("updated_at")
    df = pd.DataFrame(rows)
    df["time_period"]  = period
    df["last_updated"] = last_updated
    df["fetched_at"]   = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    return df

frames = []
for period in PERIODS:
    print(f"Fetching {period}...", end=" ")
    try:
        df = fetch_approach_period(period)
        frames.append(df)
        print(f"{len(df)} rows")
    except Exception as e:
        print(f"failed -- {e}")

if frames:
    combined = pd.concat(frames, ignore_index=True)
    meta     = ["time_period", "last_updated", "fetched_at"]
    combined = combined[meta + [c for c in combined.columns if c not in meta]]
    combined.to_csv(OUTPUT_PATH, index=False)
    print(f"[APPROACH] {len(combined):,} rows -> {OUTPUT_PATH.name}")


Fetching l12... 373 rows
Fetching l24... 384 rows
Fetching ytd... 246 rows
[APPROACH] 1,003 rows -> approach_skill_all_periods.csv
